# Taxonomy Studio (Stage 2)

This notebook runs **after a human has reviewed and approved** the draft
taxonomy YAML produced by Stage 1 (`notebooks/cleaning.ipynb`).

**Requirements to execute this notebook:**
- `OPENROUTER_API_KEY` must be set in the environment (used to build the
  OpenRouter client for per-call classification).
- `cleaned_data/taxonomies/objection_types.yaml` and `weakness_types.yaml`
  must already exist and reflect the human-reviewed/edited taxonomy (not the
  raw Stage-1 draft).

Given a human review gate on the taxonomy, this notebook:
1. Loads the frozen taxonomy into `objection_types` / `weakness_types`.
2. Classifies every call against that taxonomy via the LLM, writing
   `call_objections` / `call_weaknesses`.
3. Refreshes summary tables, exports per-rep profile YAML, and records an
   `export_meta` row (git SHA, model, row counts).
4. Prints the evaluation report and the team weakness ranking.

This notebook has **not been executed** — there is no `OPENROUTER_API_KEY` in
this environment and no reviewed taxonomy YAML yet (Stage 1 Cell 4 has not
been run with a key). Outputs are intentionally empty; do not treat any
printed values below as real.

In [ ]:
import sys, json, subprocess
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import yaml
import cleaned_data as cd
from cleaned_data import db, clustering, evaluate, cleaning_utils as cu

conn = db.connect(cd.DB_PATH)
obj = yaml.safe_load((cd.TAXONOMY_DIR / "objection_types.yaml").read_text("utf-8"))
weak = yaml.safe_load((cd.TAXONOMY_DIR / "weakness_types.yaml").read_text("utf-8"))
conn.execute("DELETE FROM objection_types"); conn.execute("DELETE FROM weakness_types")
for o in obj:
    conn.execute("INSERT INTO objection_types(obj_id,label,definition,aliases) "
                 "VALUES(?,?,?,?)", (o["id"], o["label"], o["definition"],
                                     json.dumps(o.get("aliases", []))))
for w in weak:
    conn.execute("INSERT INTO weakness_types(weak_id,label,definition,coaching_fix) "
                 "VALUES(?,?,?,?)", (w["id"], w["label"], w["definition"],
                                     w["coaching_fix"]))
conn.commit()
print("taxonomy loaded:", len(obj), "objections,", len(weak), "weaknesses")

In [ ]:
from cleaned_data import embeddings
llm = embeddings.make_client()  # OpenRouter client, reused for classification
obj_tax = [{"id": o["id"], "label": o["label"]} for o in obj]
weak_tax = [{"id": w["id"], "label": w["label"]} for w in weak]

conn.execute("DELETE FROM call_objections"); conn.execute("DELETE FROM call_weaknesses")
for c in conn.execute("SELECT call_id, objections_surfaced, what_to_improve, "
                      "why_no_close, red_flags FROM calls").fetchall():
    text = " | ".join(filter(None, [c["objections_surfaced"],
        cu.pool_weakness_text(dict(c))]))
    if not text.strip():
        continue
    res = clustering.classify_call(text, obj_tax + weak_tax, llm)
    for wid in res["weakness_ids"]:
        if any(w["id"] == wid for w in weak):
            conn.execute("INSERT INTO call_weaknesses(call_id,weak_id,evidence_quote)"
                         " VALUES(?,?,?)", (c["call_id"], wid, text[:200]))
    for o_ in res["objections"]:
        if any(t["id"] == o_["obj_id"] for t in obj):
            conn.execute("INSERT INTO call_objections(call_id,obj_id,handled,quote)"
                         " VALUES(?,?,?,?)", (c["call_id"], o_["obj_id"],
                                              o_["handled"], o_["quote"]))
conn.commit()
print("classified:", conn.execute("SELECT COUNT(*) FROM call_weaknesses").fetchone()[0],
      "weakness links")

In [ ]:
db.refresh_summary_tables(conn)
n = db.export_profiles(conn, cd.PROFILES_DIR)
sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                     text=True).stdout.strip()
counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
          for t in ["reps", "calls", "call_weaknesses", "call_objections"]}
conn.execute("INSERT INTO export_meta(generated_at,taxonomy_version,model_used,"
             "git_sha,row_counts_json) VALUES(datetime('now'),?,?,?,?)",
             ("v1", "openai/gpt-4o-mini", sha, json.dumps(counts)))
conn.commit()
print(f"exported {n} profiles")

In [ ]:
profiles = [db.build_profile_dict(conn, r[0]) for r in
            conn.execute("SELECT slug FROM reps")]
print("QUALITY:", evaluate.evaluate_profiles(profiles))
print("\nTEAM WEAKNESS RANKING:")
for row in conn.execute("SELECT wt.label, twr.rep_count, twr.call_count "
                        "FROM team_weakness_ranking twr "
                        "JOIN weakness_types wt USING(weak_id) "
                        "ORDER BY twr.call_count DESC"):
    print(f"  {row['call_count']:5d} calls / {row['rep_count']:3d} reps  {row['label']}")